# 09 — AI/BI Dashboard (Lakeview API)

Create a published Lakeview dashboard from the gold star schema — fully via API
so it's reproducible across workspaces.

## Why this matters for WAF

| WAF pillar | What this notebook proves |
|------------|---------------------------|
| **Interoperability & Usability** | The star schema + comments + tags from earlier notebooks let AI/BI auto-suggest visualizations correctly. This is what gold-layer modeling *buys* you. |
| **Operational Excellence** | Dashboard lives in source control as a JSON artifact, so rebuilds are idempotent — we even update-in-place if the dashboard already exists. |
| **Data Governance** | Published dashboards inherit UC permissions; the column mask from notebook 05 is automatically honored in widgets. |


In [1]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import Dashboard
from dotenv import load_dotenv
import os, json

load_dotenv()

w = WorkspaceClient()

UC_CATALOG       = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA        = os.getenv("UC_SCHEMA", "mlb_gumbo_waf")
GOLD_SCHEMA      = f"{UC_SCHEMA}_gold"
SQL_WAREHOUSE_ID = os.getenv("SQL_WAREHOUSE_ID")

if not SQL_WAREHOUSE_ID:
    raise ValueError("SQL_WAREHOUSE_ID not set in .env")

G = f"{UC_CATALOG}.{GOLD_SCHEMA}"
print(f"Gold schema: {G}")
print(f"Warehouse:   {SQL_WAREHOUSE_ID}")


Gold schema: alexander_booth.mlb_gumbo_waf_gold
Warehouse:   49458fa9801b109b


In [2]:
DASHBOARD_NAME = "MLB GUMBO — Pitch Analytics (WAF)"

datasets = [
    {"name": "kpi_summary", "displayName": "Season KPIs",
     "query": f"SELECT COUNT(DISTINCT game_sk) AS games, COUNT(*) AS pitches, ROUND(AVG(start_speed_mph),2) AS avg_velocity_mph, ROUND(100.0*SUM(CASE WHEN is_strike THEN 1 ELSE 0 END)/COUNT(*),2) AS strike_pct FROM {G}.fact_pitches"},
    {"name": "pitch_mix", "displayName": "Pitch Mix by Type",
     "query": f"SELECT * FROM {G}.v_pitch_mix_by_type ORDER BY n_pitches DESC"},
    {"name": "pitcher_leaderboard", "displayName": "Top Pitchers",
     "query": f"SELECT * FROM {G}.v_pitcher_leaderboard ORDER BY avg_velocity_mph DESC LIMIT 25"},
    {"name": "team_pitching", "displayName": "Team Pitching Summary",
     "query": f"SELECT * FROM {G}.v_team_pitching_summary ORDER BY pitches DESC"},
    {"name": "velocity_by_date", "displayName": "Avg Velocity by Game Date",
     "query": f"SELECT official_date, ROUND(AVG(start_speed_mph),2) AS avg_velocity_mph, COUNT(*) AS pitches FROM {G}.fact_pitches WHERE official_date IS NOT NULL GROUP BY 1 ORDER BY 1"},
    {"name": "strike_prob_calibration", "displayName": "Model Strike Prob Calibration",
     "query": f"""
        SELECT ROUND(called_strike_prob, 1) AS bucket,
               COUNT(*) AS n
        FROM {G}.pitch_strike_probability
        GROUP BY 1
        ORDER BY 1""".strip()},
]

dashboard_json = {
    "pages": [{"name": "pitch_overview", "displayName": "MLB Pitch Overview"}],
    "datasets": datasets,
}
serialized = json.dumps(dashboard_json)
print(f"Serialized dashboard: {len(serialized):,} chars")
print(f"Datasets: {len(datasets)}")
for d in datasets:
    print(f"  - {d['displayName']}")


Serialized dashboard: 1,503 chars
Datasets: 6
  - Season KPIs
  - Pitch Mix by Type
  - Top Pitchers
  - Team Pitching Summary
  - Avg Velocity by Game Date
  - Model Strike Prob Calibration


In [3]:
current_user = w.current_user.me()
parent_path = f"/Workspace/Users/{current_user.user_name}"

existing_id = None
try:
    for obj in w.workspace.list(parent_path):
        if obj.path and obj.path.endswith(f"{DASHBOARD_NAME}.lvdash.json"):
            existing_id = str(obj.object_id)
            break
except Exception:
    pass

if existing_id:
    try:
        d = w.lakeview.get(dashboard_id=existing_id)
        w.lakeview.update(
            dashboard_id=d.dashboard_id,
            dashboard=Dashboard(serialized_dashboard=serialized, warehouse_id=SQL_WAREHOUSE_ID),
        )
        dashboard_id = d.dashboard_id
        print(f"Dashboard updated in place: {dashboard_id}")
    except Exception:
        existing_id = None

if not existing_id:
    d = w.lakeview.create(
        dashboard=Dashboard(
            display_name=DASHBOARD_NAME,
            warehouse_id=SQL_WAREHOUSE_ID,
            serialized_dashboard=serialized,
            parent_path=parent_path,
        )
    )
    dashboard_id = d.dashboard_id
    print(f"Dashboard created: {dashboard_id}")

# Publish so consumers (Databricks One) can see it
w.lakeview.publish(dashboard_id=dashboard_id, warehouse_id=SQL_WAREHOUSE_ID, embed_credentials=True)
host = os.getenv("DATABRICKS_HOST", "").rstrip("/")
print(f"\nView at: {host}/sql/dashboardsv3/{dashboard_id}")


Dashboard updated in place: 01f13dbc8aa115209e98d6685962b559



View at: https://e2-demo-field-eng.cloud.databricks.com/sql/dashboardsv3/01f13dbc8aa115209e98d6685962b559
